In [1]:
import nbloss

print("Imported from:", nbloss.__file__)

Imported from: C:\Users\koeng\Downloads\nbloss-main\nbloss-main\nbloss\__init__.py


# Step 1 — Load and inspect the Framingham dataset

# Framingham Data Preparation

This script constructs the dataset used for the 15-year Framingham ANYCHD prediction experiment.

Starting from the original longitudinal Framingham dataset, it:

- Retains the baseline examination (`PERIOD == 1`).
- Excludes participants with prevalent coronary heart disease at baseline.
- Defines incident ANYCHD within 15 years as the binary outcome.
- Excludes participants whose 15-year outcome cannot be determined because of insufficient follow-up.
- Selects the baseline predictors used in the modelling experiments.
- Creates the final modelling dataset without participant identifiers.

The resulting dataset is used to compare conventional NLL/BCE training with Smooth Net Benefit training. 

In [ ]:
# ============================================================
# Step 1 — Load and prepare Framingham data
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Load original longitudinal Framingham dataset
# ------------------------------------------------------------

DATA_PATH = Path("Data") / "framingham.csv"

df_raw = pd.read_csv(
    DATA_PATH,
    sep=None,
    engine="python",
)

df_raw = df_raw.drop(
    columns=["Unnamed: 0"],
    errors="ignore",
)


# ------------------------------------------------------------
# Keep baseline examination only
# ------------------------------------------------------------

df = df_raw.loc[
    df_raw["PERIOD"] == 1
].copy()

assert df["RANDID"].is_unique


# ------------------------------------------------------------
# Exclude prevalent coronary heart disease
# ------------------------------------------------------------

df = df.loc[
    df["PREVCHD"] == 0
].copy()

assert len(df) == 4240


# ------------------------------------------------------------
# Construct 15-year incident ANYCHD outcome
# ------------------------------------------------------------

HORIZON_YEARS = 15
HORIZON_DAYS = 365.25 * HORIZON_YEARS

event_15y = (
    (df["ANYCHD"] == 1)
    & (df["TIMECHD"] <= HORIZON_DAYS)
)

non_event_15y = (
    ~event_15y
    & (df["TIMECHD"] >= HORIZON_DAYS)
)

known_outcome = (
    event_15y
    | non_event_15y
)

df = df.loc[
    known_outcome
].copy()

df["FifteenYearCHD"] = (
    event_15y.loc[df.index]
    .astype(int)
)


# ------------------------------------------------------------
# Retain modelling variables
# ------------------------------------------------------------

PREDICTOR_COLS = [
    "SEX",
    "TOTCHOL",
    "AGE",
    "SYSBP",
    "DIABP",
    "CURSMOKE",
    "CIGPDAY",
    "BMI",
    "DIABETES",
    "BPMEDS",
    "HEARTRTE",
    "GLUCOSE",
    "educ",
    "PREVSTRK",
    "PREVHYP",
]

TARGET_COL = "FifteenYearCHD"

model_df = df[
    PREDICTOR_COLS + [TARGET_COL]
].copy()


# ------------------------------------------------------------
# Ensure modelling variables are numeric
# ------------------------------------------------------------

for col in model_df.columns:
    model_df[col] = pd.to_numeric(
        model_df[col]
        .astype(str)
        .str.replace(",", ".", regex=False),
        errors="coerce",
    )


# ------------------------------------------------------------
# Basic checks
# ------------------------------------------------------------

assert model_df[TARGET_COL].isin([0, 1]).all()

print("Modelling dataset shape:", model_df.shape)
print(
    "15-year ANYCHD prevalence:",
    f"{model_df[TARGET_COL].mean():.3%}",
)

print("\nMissing values:")
print(
    model_df
    .isna()
    .sum()
    .sort_values(ascending=False)
)

# Step 2 — Create repeated train/test splits and preprocess the data

This step creates the five rotating 80/20 train/test splits used for the Framingham generalized additive model (GAM) experiment. All preprocessing steps are fitted on the training data only and then applied to the corresponding test fold. No validation set is used. Two preprocessing variants are available: `mild`, where education is retained as a single ordinal variable, and `moderate`, where education is one-hot encoded. In both cases, continuous variables are transformed into spline basis expansions to allow the GAM to model nonlinear effects.

In [ ]:
from dataclasses import dataclass

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, SplineTransformer


# ============================================================
# Framingham GAM preprocessing
# New 15-year incident ANYCHD analysis
# ============================================================


# ============================================================
# Variable definitions
#
# These deliberately match the NEW LR preprocessing exactly.
# ============================================================

FRAM_BINARY_COLS = [
    "SEX",
    "CURSMOKE",
    "DIABETES",
    "BPMEDS",
    "PREVSTRK",
    "PREVHYP",
]

FRAM_CATEGORICAL_COLS = [
    "educ",
]

FRAM_CONTINUOUS_COLS = [
    "AGE",
    "CIGPDAY",
    "TOTCHOL",
    "SYSBP",
    "DIABP",
    "BMI",
    "HEARTRTE",
    "GLUCOSE",
]


# ============================================================
# Preprocessor bundle
# ============================================================

@dataclass
class FraminghamGAMPreprocessorBundle:
    fe_level: str

    binary_cols: list
    categorical_cols: list
    continuous_cols: list

    imputer_bin: SimpleImputer
    imputer_cont: SimpleImputer
    scaler_cont: StandardScaler
    spline: SplineTransformer

    education_cols: list
    base_feature_names: list
    gam_feature_names: list

    education_dim: int
    spline_dim: int
    bin_dim: int

    spline_blocks: dict
    spline_block_slices: list
    non_spline_idx: np.ndarray

    imputer_edu: object = None


# ============================================================
# Helpers
# ============================================================

def _to_float32(x):
    return np.asarray(x, dtype=np.float32)


def _validate_framingham_gam_columns(
    df: pd.DataFrame,
    target_col: str,
):
    required = set(
        FRAM_BINARY_COLS
        + FRAM_CATEGORICAL_COLS
        + FRAM_CONTINUOUS_COLS
        + [target_col]
    )

    missing = [
        c for c in required
        if c not in df.columns
    ]

    if missing:
        raise ValueError(
            f"Missing required columns: {missing}"
        )


def _prepare_framingham_gam_frame(
    df: pd.DataFrame,
    target_col: str,
) -> pd.DataFrame:

    cols = (
        FRAM_BINARY_COLS
        + FRAM_CATEGORICAL_COLS
        + FRAM_CONTINUOUS_COLS
        + [target_col]
    )

    return df[cols].copy()


# ============================================================
# Education preprocessing
# Mild:
#   single numeric/ordinal education column
# ============================================================

def _education_no_ohe_train_test_gam(
    X_train_cat: pd.DataFrame,
    X_test_cat: pd.DataFrame,
):

    imputer_edu = SimpleImputer(
        strategy="most_frequent"
    )

    X_train_edu = pd.DataFrame(
        imputer_edu.fit_transform(
            X_train_cat[["educ"]]
        ),
        columns=["educ"],
        index=X_train_cat.index,
    ).astype(float)

    X_test_edu = pd.DataFrame(
        imputer_edu.transform(
            X_test_cat[["educ"]]
        ),
        columns=["educ"],
        index=X_test_cat.index,
    ).astype(float)

    return (
        X_train_edu,
        X_test_edu,
        ["educ"],
        imputer_edu,
    )


# ============================================================
# Education preprocessing
# Moderate:
#   one-hot encoded education
#
# This deliberately matches the NEW LR preprocessing:
# drop_first=False
# missing education becomes a separate category
# ============================================================

def _ohe_education_train_test_gam(
    X_train_cat: pd.DataFrame,
    X_test_cat: pd.DataFrame,
):

    X_train_cat = X_train_cat.copy()
    X_test_cat = X_test_cat.copy()

    for frame in (
        X_train_cat,
        X_test_cat,
    ):
        frame["educ"] = (
            frame["educ"]
            .astype("object")
            .where(
                ~frame["educ"].isna(),
                "missing",
            )
        )

        frame["educ"] = (
            frame["educ"]
            .astype(str)
        )

    train_ohe = pd.get_dummies(
        X_train_cat,
        columns=["educ"],
        prefix="educ",
        drop_first=False,
    )

    test_ohe = pd.get_dummies(
        X_test_cat,
        columns=["educ"],
        prefix="educ",
        drop_first=False,
    )

    # Training categories determine the final feature space.
    ohe_cols = train_ohe.columns.tolist()

    test_ohe = test_ohe.reindex(
        columns=ohe_cols,
        fill_value=0.0,
    )

    train_ohe = train_ohe.astype(float)
    test_ohe = test_ohe.astype(float)

    return (
        train_ohe,
        test_ohe,
        ohe_cols,
    )


# ============================================================
# Fit preprocessing on TRAIN only, then transform TRAIN + TEST
# ============================================================

def fit_train_preprocessor_framingham_gam_and_transform(
    X_tr: pd.DataFrame,
    X_te: pd.DataFrame,
    *,
    fe_level: str = "mild",
    spline_n_knots: int = 8,
    spline_degree: int = 3,
):
    """
    Framingham-specific GAM preprocessing for the new
    15-year ANYCHD analysis.

    Raw predictors are identical to the logistic-regression
    analysis.

    BASE MATRIX
    -----------

    Mild:
        [educ | scaled continuous | binary]

    Moderate:
        [OHE educ | scaled continuous | binary]


    GAM MATRIX
    ----------

    Mild:
        [educ | spline(scaled continuous) | binary]

    Moderate:
        [OHE educ | spline(scaled continuous) | binary]


    All imputers, scaling parameters, OHE levels and spline
    transformations are fitted exclusively on the training data.
    """

    if fe_level not in (
        "mild",
        "moderate",
    ):
        raise ValueError(
            "fe_level must be 'mild' or 'moderate'"
        )

    X_tr = X_tr.copy()
    X_te = X_te.copy()

    # ========================================================
    # Split variables by type
    # ========================================================

    X_tr_bin_raw = (
        X_tr[FRAM_BINARY_COLS]
        .copy()
    )

    X_te_bin_raw = (
        X_te[FRAM_BINARY_COLS]
        .copy()
    )

    X_tr_cat = (
        X_tr[FRAM_CATEGORICAL_COLS]
        .copy()
    )

    X_te_cat = (
        X_te[FRAM_CATEGORICAL_COLS]
        .copy()
    )

    X_tr_cont = (
        X_tr[FRAM_CONTINUOUS_COLS]
        .copy()
    )

    X_te_cont = (
        X_te[FRAM_CONTINUOUS_COLS]
        .copy()
    )


    # ========================================================
    # Binary variables
    #
    # Same as LR:
    # mode imputation, no scaling
    # ========================================================

    imputer_bin = SimpleImputer(
        strategy="most_frequent"
    )

    X_tr_bin = pd.DataFrame(
        imputer_bin.fit_transform(
            X_tr_bin_raw
        ),
        columns=FRAM_BINARY_COLS,
        index=X_tr_bin_raw.index,
    ).astype(float)

    X_te_bin = pd.DataFrame(
        imputer_bin.transform(
            X_te_bin_raw
        ),
        columns=FRAM_BINARY_COLS,
        index=X_te_bin_raw.index,
    ).astype(float)


    # ========================================================
    # Education
    #
    # Identical encoding to LR.
    # ========================================================

    if fe_level == "mild":

        (
            X_tr_edu,
            X_te_edu,
            education_cols,
            imputer_edu,
        ) = _education_no_ohe_train_test_gam(
            X_tr_cat,
            X_te_cat,
        )

    else:

        (
            X_tr_edu,
            X_te_edu,
            education_cols,
        ) = _ohe_education_train_test_gam(
            X_tr_cat,
            X_te_cat,
        )

        imputer_edu = None


    # ========================================================
    # Continuous variables
    #
    # Same first steps as LR:
    #   1. median imputation
    #   2. standardization
    #
    # The standardized variables are subsequently expanded
    # into spline basis functions for GAM.
    # ========================================================

    imputer_cont = SimpleImputer(
        strategy="median"
    )

    scaler_cont = StandardScaler()

    # Fit imputation on TRAIN only
    X_tr_cont_i = pd.DataFrame(
        imputer_cont.fit_transform(
            X_tr_cont
        ),
        columns=FRAM_CONTINUOUS_COLS,
        index=X_tr_cont.index,
    )

    X_te_cont_i = pd.DataFrame(
        imputer_cont.transform(
            X_te_cont
        ),
        columns=FRAM_CONTINUOUS_COLS,
        index=X_te_cont.index,
    )

    # Fit scaling on TRAIN only
    X_tr_cont_s = pd.DataFrame(
        scaler_cont.fit_transform(
            X_tr_cont_i
        ),
        columns=FRAM_CONTINUOUS_COLS,
        index=X_tr_cont_i.index,
    )

    X_te_cont_s = pd.DataFrame(
        scaler_cont.transform(
            X_te_cont_i
        ),
        columns=FRAM_CONTINUOUS_COLS,
        index=X_te_cont_i.index,
    )


    # ========================================================
    # Base matrix
    #
    # This provides the LR-like representation for diagnostics
    # and preserves compatibility with the existing GAM code.
    # ========================================================

    X_tr_base_df = pd.concat(
        [
            X_tr_edu,
            X_tr_cont_s,
            X_tr_bin,
        ],
        axis=1,
    )

    X_te_base_df = pd.concat(
        [
            X_te_edu,
            X_te_cont_s,
            X_te_bin,
        ],
        axis=1,
    )

    base_feature_names = (
        X_tr_base_df
        .columns
        .tolist()
    )

    assert (
        X_tr_base_df.columns.tolist()
        == X_te_base_df.columns.tolist()
    )


    # ========================================================
    # Spline expansion
    #
    # ONLY continuous variables receive splines.
    #
    # Education remains linear/categorical.
    # Binary predictors remain linear binary terms.
    # ========================================================

    spline = SplineTransformer(
        n_knots=int(spline_n_knots),
        degree=int(spline_degree),
        include_bias=False,
    )

    # Fit spline basis on TRAIN only
    Phi_tr_num = spline.fit_transform(
        X_tr_cont_s
    )

    Phi_te_num = spline.transform(
        X_te_cont_s
    )

    Phi_tr_num = pd.DataFrame(
        Phi_tr_num,
        index=X_tr_cont_s.index,
    )

    Phi_te_num = pd.DataFrame(
        Phi_te_num,
        index=X_te_cont_s.index,
    )


    # ========================================================
    # Determine spline dimensions
    # ========================================================

    total_spline_dim = int(
        Phi_tr_num.shape[1]
    )

    n_num = len(
        FRAM_CONTINUOUS_COLS
    )

    if total_spline_dim % n_num != 0:
        raise RuntimeError(
            "Unexpected spline output dimension; "
            "cannot form equal blocks."
        )

    per_feature_dim = (
        total_spline_dim // n_num
    )


    # ========================================================
    # Give spline columns informative names
    # ========================================================

    spline_feature_names = []

    for col in FRAM_CONTINUOUS_COLS:
        for j in range(per_feature_dim):
            spline_feature_names.append(
                f"{col}_spline_{j}"
            )

    Phi_tr_num.columns = (
        spline_feature_names
    )

    Phi_te_num.columns = (
        spline_feature_names
    )


    # ========================================================
    # Final GAM matrix
    #
    # Order:
    #   education
    #   spline-expanded continuous predictors
    #   binary predictors
    #
    # This ordering is important because spline_blocks and
    # non_spline_idx rely on it.
    # ========================================================

    Phi_tr_df = pd.concat(
        [
            X_tr_edu,
            Phi_tr_num,
            X_tr_bin,
        ],
        axis=1,
    )

    Phi_te_df = pd.concat(
        [
            X_te_edu,
            Phi_te_num,
            X_te_bin,
        ],
        axis=1,
    )

    assert (
        Phi_tr_df.columns.tolist()
        == Phi_te_df.columns.tolist()
    )

    gam_feature_names = (
        Phi_tr_df
        .columns
        .tolist()
    )


    # ========================================================
    # Matrix dimensions
    # ========================================================

    education_dim = len(
        education_cols
    )

    spline_dim = int(
        Phi_tr_num.shape[1]
    )

    bin_dim = len(
        FRAM_BINARY_COLS
    )


    # ========================================================
    # Build spline index blocks
    #
    # Used later for spline-specific penalties / GAM structure.
    # ========================================================

    spline_start = (
        education_dim
    )

    spline_blocks = {}
    spline_block_slices = []

    for j, col in enumerate(
        FRAM_CONTINUOUS_COLS
    ):

        s0 = (
            spline_start
            + j * per_feature_dim
        )

        s1 = (
            s0
            + per_feature_dim
        )

        spline_blocks[col] = list(
            range(s0, s1)
        )

        spline_block_slices.append(
            (col, s0, s1)
        )


    # ========================================================
    # Indices for NON-spline coefficients
    #
    # These are:
    #   education terms
    #   binary terms
    #
    # Continuous spline coefficients are excluded.
    # ========================================================

    non_spline_idx = np.array(
        list(
            range(
                education_dim
            )
        )
        +
        list(
            range(
                education_dim + spline_dim,
                education_dim + spline_dim + bin_dim,
            )
        ),
        dtype=np.int64,
    )


    # ========================================================
    # Convert to float32 arrays
    # ========================================================

    X_tr_base = _to_float32(
        X_tr_base_df.to_numpy()
    )

    X_te_base = _to_float32(
        X_te_base_df.to_numpy()
    )

    Phi_tr = _to_float32(
        Phi_tr_df.to_numpy()
    )

    Phi_te = _to_float32(
        Phi_te_df.to_numpy()
    )


    # ========================================================
    # Integrity checks
    # ========================================================

    if np.isnan(X_tr_base).any():
        raise RuntimeError(
            "NaNs found in X_tr_base."
        )

    if np.isnan(X_te_base).any():
        raise RuntimeError(
            "NaNs found in X_te_base."
        )

    if np.isnan(Phi_tr).any():
        raise RuntimeError(
            "NaNs found in Phi_tr."
        )

    if np.isnan(Phi_te).any():
        raise RuntimeError(
            "NaNs found in Phi_te."
        )

    if Phi_tr.shape[1] != (
        education_dim
        + spline_dim
        + bin_dim
    ):
        raise RuntimeError(
            "Unexpected GAM matrix dimension."
        )


    # ========================================================
    # Store fitted preprocessing objects
    # ========================================================

    bundle = FraminghamGAMPreprocessorBundle(
        fe_level=fe_level,

        binary_cols=FRAM_BINARY_COLS.copy(),
        categorical_cols=(
            FRAM_CATEGORICAL_COLS.copy()
        ),
        continuous_cols=(
            FRAM_CONTINUOUS_COLS.copy()
        ),

        imputer_bin=imputer_bin,
        imputer_cont=imputer_cont,
        scaler_cont=scaler_cont,
        spline=spline,

        education_cols=education_cols,

        base_feature_names=(
            base_feature_names
        ),

        gam_feature_names=(
            gam_feature_names
        ),

        education_dim=education_dim,
        spline_dim=spline_dim,
        bin_dim=bin_dim,

        spline_blocks=spline_blocks,

        spline_block_slices=(
            spline_block_slices
        ),

        non_spline_idx=(
            non_spline_idx
        ),

        imputer_edu=imputer_edu,
    )


    # ========================================================
    # Return same general structure as previous GAM pipeline
    # ========================================================

    return {
        "bundle": bundle,
        "enc": bundle,

        "X_tr_base": X_tr_base,
        "X_te_base": X_te_base,

        "Phi_tr": Phi_tr,
        "Phi_te": Phi_te,

        "binary_cols": (
            bundle.binary_cols
        ),

        "categorical_cols": (
            bundle.categorical_cols
        ),

        "numeric_cols": (
            bundle.continuous_cols
        ),

        "spline_blocks": (
            bundle.spline_blocks
        ),

        "spline_block_slices": (
            bundle.spline_block_slices
        ),

        "non_spline_idx": (
            bundle.non_spline_idx
        ),

        "base_feature_names": (
            bundle.base_feature_names
        ),

        "gam_feature_names": (
            bundle.gam_feature_names
        ),

        "scaler": scaler_cont,
        "spline": spline,
    }


# ============================================================
# Create 5 rotating stratified GAM folds
# ============================================================

def make_framingham_gam_splits_5x(
    df: pd.DataFrame,
    *,
    target_col: str = "FifteenYearCHD",
    base_seed: int = 42,
    n_runs: int = 5,
    fe_level: str = "mild",
    spline_n_knots: int = 8,
    spline_degree: int = 3,
):

    if n_runs != 5:
        raise ValueError(
            "This regime uses exactly 5 rotating folds, "
            "so n_runs must be 5."
        )

    if fe_level not in (
        "mild",
        "moderate",
    ):
        raise ValueError(
            "fe_level must be 'mild' or 'moderate'."
        )


    # ========================================================
    # Validate + restrict to relevant columns
    # ========================================================

    _validate_framingham_gam_columns(
        df,
        target_col,
    )

    df_base = (
        _prepare_framingham_gam_frame(
            df,
            target_col,
        )
    )


    # ========================================================
    # Outcome
    # ========================================================

    y_all = (
        df_base[target_col]
        .to_numpy(dtype=np.float32)
        .reshape(-1)
    )

    if not np.isin(
        y_all,
        [0, 1],
    ).all():
        raise ValueError(
            "Target must contain only 0 and 1."
        )


    # ========================================================
    # Predictors
    # ========================================================

    X_all = (
        df_base
        .drop(columns=[target_col])
        .copy()
    )


    # ========================================================
    # Same stratified folds as LR
    # ========================================================

    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=int(base_seed),
    )

    splits = []


    # ========================================================
    # Loop over rotating test folds
    # ========================================================

    for run_id, (
        train_idx,
        test_idx,
    ) in enumerate(
        skf.split(
            X_all,
            y_all.astype(int),
        )
    ):

        split_seed = (
            int(base_seed)
            + int(run_id)
        )

        X_train_df = (
            X_all
            .iloc[train_idx]
            .copy()
        )

        X_test_df = (
            X_all
            .iloc[test_idx]
            .copy()
        )

        y_train = (
            y_all[train_idx]
            .astype(np.float32)
        )

        y_test = (
            y_all[test_idx]
            .astype(np.float32)
        )

        # Fit preprocessing on training fold only
        pack = (
            fit_train_preprocessor_framingham_gam_and_transform(
                X_train_df,
                X_test_df,
                fe_level=fe_level,
                spline_n_knots=spline_n_knots,
                spline_degree=spline_degree,
            )
        )

        splits.append(
            {
                "run_id": int(run_id),

                "seed": int(
                    split_seed
                ),

                "fe_level": (
                    fe_level
                ),

                "X_train_df": (
                    X_train_df
                ),

                "X_test_df": (
                    X_test_df
                ),

                "y_train": (
                    y_train
                ),

                "y_test": (
                    y_test
                ),

                "X_tr_base": (
                    pack["X_tr_base"]
                ),

                "X_te_base": (
                    pack["X_te_base"]
                ),

                "Phi_tr": (
                    pack["Phi_tr"]
                ),

                "Phi_te": (
                    pack["Phi_te"]
                ),

                "bundle": (
                    pack["bundle"]
                ),

                "enc": (
                    pack["enc"]
                ),

                "binary_cols": (
                    pack["binary_cols"]
                ),

                "categorical_cols": (
                    pack["categorical_cols"]
                ),

                "numeric_cols": (
                    pack["numeric_cols"]
                ),

                "spline_blocks": (
                    pack["spline_blocks"]
                ),

                "spline_block_slices": (
                    pack["spline_block_slices"]
                ),

                "non_spline_idx": (
                    pack["non_spline_idx"]
                ),

                "base_feature_names": (
                    pack["base_feature_names"]
                ),

                "gam_feature_names": (
                    pack["gam_feature_names"]
                ),

                "prev_train": float(
                    y_train.mean()
                ),

                "prev_test": float(
                    y_test.mean()
                ),
            }
        )

    return splits

# Step 3 — Configure the GAM experiment

This step defines the global settings for the Framingham generalized additive model (GAM) experiment. We set the computation device, random seeds, smooth Net Benefit annealing schedule, threshold-band policy, and the simple GAM model used for both BCE training and subsequent SNB fine-tuning. The GAM extends logistic regression by replacing the continuous predictors with spline basis expansions, allowing nonlinear effects while retaining an additive model structure.

In [9]:
import random

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from nbloss.metrics import average_nb_over_range
from nbloss.trainer import make_optimizer, set_seed, nb_anneal_only_with_l2 


DEVICE = "cuda" if torch.cuda.is_available() else (
    "mps" if getattr(torch.backends, "mps", None)
    and torch.backends.mps.is_available()
    else "cpu"
)

SEED_GLOBAL = 1234
MODEL_SEED = 4242

set_seed(SEED_GLOBAL)

LR_ADAMW = 3e-2
BATCH = 1024

INVERSE_TEMPS = (1.0, 4.0, 10.0)
EPOCHS_PER_TEMP = 300
PATIENCE_HARD = 20
TRAIN_RANGE_POINTS = 11
TEST_RANGE_POINTS = 201

BAND_HALF_WIDTH = 0.025

LOCAL_START_HALF_WIDTH = 0.1
LOCAL_EXPAND_STEP = 0.01
LOCAL_MIN_POS = 50
LOCAL_MIN_NEG = 50 

TEMP_MAX_ITER = 500
PLATT_MAX_ITER = 500


def band_from_t_ref(
    t_ref: float,
    half_width: float = BAND_HALF_WIDTH,
) -> tuple[float, float]:
    eps = 1e-9
    t_ref = float(np.clip(t_ref, eps, 1.0 - eps))

    t_min = max(eps, t_ref - float(half_width))
    t_max = min(1.0 - eps, t_ref + float(half_width))

    if not t_min < t_max:
        span = min(t_ref - eps, 1.0 - eps - t_ref, float(half_width))
        t_min = t_ref - span
        t_max = t_ref + span

    return float(t_min), float(t_max)


def make_loader(
    X,
    y,
    batch: int = BATCH,
    shuffle: bool = False,
    seed: int = SEED_GLOBAL,
):
    g = torch.Generator().manual_seed(int(seed))

    ds = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32).view(-1),
    )

    return DataLoader(
        ds,
        batch_size=int(batch),
        shuffle=bool(shuffle),
        generator=g,
        drop_last=False,
    )


class TorchGAM(nn.Module):
    def __init__(self, d_in_phi: int):
        super().__init__()
        self.linear = nn.Linear(d_in_phi, 1, bias=True)

    def forward(self, phi_x):
        return self.linear(phi_x).squeeze(-1)


def optimizer_factory_no_wd(model: torch.nn.Module):
    # Explicit regularization penalties are added to the loss,
    # so optimizer weight decay remains disabled.
    return make_optimizer(
        model,
        name="adamw",
        lr=LR_ADAMW,
        weight_decay=0.0,
    )

# Step 4 — Define data loaders and helper losses

This step defines small helper functions used by the generalized additive model (GAM) training code. The data loader keeps the outcome as a one-dimensional tensor to match the model output shape and is used to feed the spline-expanded feature matrix to the model during mini-batch optimization. The L2 penalty is applied only to model weights and excludes bias terms, matching the benchmark protocol.

In [10]:
from torch.utils.data import DataLoader, TensorDataset


def make_loader(
    X: np.ndarray,
    y: np.ndarray,
    *,
    batch: int = 1024,
    shuffle: bool = False,
    seed: int = 1234,
) -> DataLoader:
    g = torch.Generator().manual_seed(int(seed))

    ds = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32).view(-1),
    )

    return DataLoader(
        ds,
        batch_size=int(batch),
        shuffle=bool(shuffle),
        generator=g,
        drop_last=False,
    )


def l2_penalty_weights_only(model: nn.Module) -> torch.Tensor:
    device = next(model.parameters()).device
    l2 = torch.zeros((), device=device)

    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name.endswith("bias") or name in ("bias", "b0", "intercept"):
            continue
        l2 = l2 + (p * p).sum()

    return l2


@torch.no_grad()
def bce_logits_loss(
    model: nn.Module,
    X: torch.Tensor,
    y: torch.Tensor,
) -> float:
    model.eval()
    logits = model(X).view(-1)
    y = y.view(-1)
    loss = nn.BCEWithLogitsLoss(reduction="mean")(logits, y)
    return float(loss.detach().cpu().item())

# Step 5 — Define BCE-GAM fitting and penalty-selection helpers

This step defines the PyTorch helpers used to fit the BCE-GAM on the spline-expanded GAM design matrix. The model is fitted with LBFGS and two explicit penalties: an L2 penalty on the non-spline coefficients and a second-difference smoothness penalty within each spline block. The penalty strengths are selected by inner cross-validation on the training data only, after which the selected BCE-GAM is refitted on the full training split.

In [11]:
import numpy as np
import torch
import torch.nn as nn

from sklearn.model_selection import StratifiedKFold


def _get_weight_vector(model: nn.Module) -> torch.Tensor:
    """
    Return the 1D coefficient vector excluding intercept.
    Assumes model.linear is nn.Linear(d_in, 1, bias=True).
    """
    if not hasattr(model, "linear"):
        raise AttributeError("Expected model to have attribute 'linear'.")
    return model.linear.weight.view(-1)


def _linear_l2_penalty_from_idx(model: nn.Module, non_spline_idx) -> torch.Tensor:
    """
    Ridge penalty on NON-spline coefficients only.
    """
    w = _get_weight_vector(model)
    device = w.device

    if non_spline_idx is None or len(non_spline_idx) == 0:
        return torch.zeros((), device=device)

    idx = torch.as_tensor(non_spline_idx, dtype=torch.long, device=device)
    return (w[idx] ** 2).sum()


def _smoothness_penalty_from_blocks(model: nn.Module, spline_block_slices) -> torch.Tensor:
    """
    Second-order finite-difference penalty within each spline block.
    """
    w = _get_weight_vector(model)
    device = w.device
    pen = torch.zeros((), device=device)

    if spline_block_slices is None:
        return pen

    for _, start, end in spline_block_slices:
        beta = w[start:end]
        if beta.numel() < 3:
            continue
        d2 = beta[2:] - 2.0 * beta[1:-1] + beta[:-2]
        pen = pen + (d2 ** 2).sum()

    return pen


@torch.no_grad()
def _bce_logits_loss(model: nn.Module, X: torch.Tensor, y: torch.Tensor) -> float:
    model.eval()
    logits = model(X).view(-1)
    y = y.view(-1)
    loss = nn.BCEWithLogitsLoss(reduction="mean")(logits, y)
    return float(loss.detach().cpu().item())


def fit_bce_gam_lbfgs(
    make_model_fn,
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    *,
    lambda_lin: float,
    lambda_smooth: float,
    non_spline_idx,
    spline_block_slices,
    max_iter: int = 500,
    tol_grad: float = 1e-7,
    tol_change: float = 1e-9,
    history_size: int = 100,
    device: str = "cpu",
):
    """
    Convex BCE fit with LBFGS on a fixed GAM design matrix Phi.

    Objective:
        BCE
      + lambda_lin    * L2(non-spline coefficients)
      + lambda_smooth * smoothness_penalty(spline blocks)
    """
    device_t = torch.device(device)

    X = torch.tensor(X_tr, dtype=torch.float32, device=device_t)
    y = torch.tensor(y_tr, dtype=torch.float32, device=device_t).view(-1)

    model = make_model_fn().to(device_t)
    bce = nn.BCEWithLogitsLoss(reduction="mean")

    opt = torch.optim.LBFGS(
        model.parameters(),
        lr=1.0,
        max_iter=int(max_iter),
        tolerance_grad=float(tol_grad),
        tolerance_change=float(tol_change),
        history_size=int(history_size),
        line_search_fn="strong_wolfe",
    )

    lambda_lin_t = torch.tensor(float(lambda_lin), device=device_t)
    lambda_smooth_t = torch.tensor(float(lambda_smooth), device=device_t)

    def closure():
        opt.zero_grad(set_to_none=True)

        logits = model(X).view(-1)
        loss_bce = bce(logits, y)

        pen_lin = _linear_l2_penalty_from_idx(model, non_spline_idx)
        pen_smooth = _smoothness_penalty_from_blocks(model, spline_block_slices)

        loss = loss_bce + lambda_lin_t * pen_lin + lambda_smooth_t * pen_smooth

        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite LBFGS loss: {loss.detach().item()}")

        loss.backward()
        return loss

    opt.step(closure)

    train_bce = _bce_logits_loss(model, X, y)
    return model, train_bce


def select_gam_penalties_by_cv_bce(
    lambda_lin_grid,
    lambda_smooth_grid,
    *,
    make_model_fn,
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    non_spline_idx,
    spline_block_slices,
    device: str = "cpu",
    n_splits: int = 5,
    lbfgs_max_iter: int = 500,
    seed: int = 1234,
):
    """
    Select (lambda_lin, lambda_smooth) using INNER CV on training data.

    Returns:
        lambda_lin_star, lambda_smooth_star, best_cv_bce, table_rows
    """
    skf = StratifiedKFold(
        n_splits=int(n_splits),
        shuffle=True,
        random_state=int(seed),
    )

    device_t = torch.device(device)

    best = None
    rows = []

    for lambda_lin in lambda_lin_grid:
        for lambda_smooth in lambda_smooth_grid:
            fold_bces = []

            for fold_id, (inner_tr_idx, inner_va_idx) in enumerate(skf.split(X_tr, y_tr)):
                X_inner_tr = X_tr[inner_tr_idx]
                y_inner_tr = y_tr[inner_tr_idx]

                X_inner_va = X_tr[inner_va_idx]
                y_inner_va = y_tr[inner_va_idx]

                model, train_bce = fit_bce_gam_lbfgs(
                    make_model_fn,
                    X_inner_tr,
                    y_inner_tr,
                    lambda_lin=float(lambda_lin),
                    lambda_smooth=float(lambda_smooth),
                    non_spline_idx=non_spline_idx,
                    spline_block_slices=spline_block_slices,
                    max_iter=lbfgs_max_iter,
                    device=device,
                )

                Xva_t = torch.tensor(X_inner_va, dtype=torch.float32, device=device_t)
                yva_t = torch.tensor(y_inner_va, dtype=torch.float32, device=device_t).view(-1)

                val_bce = _bce_logits_loss(model, Xva_t, yva_t)
                fold_bces.append(float(val_bce))

            mean_cv_bce = float(np.mean(fold_bces))

            rows.append({
                "lambda_lin": float(lambda_lin),
                "lambda_smooth": float(lambda_smooth),
                "cv_bce_mean": mean_cv_bce,
                "cv_bce_folds": fold_bces,
            })

            if (best is None) or (mean_cv_bce < best[2]):
                best = (float(lambda_lin), float(lambda_smooth), mean_cv_bce)

    lambda_lin_star, lambda_smooth_star, best_cv_bce = best
    return lambda_lin_star, lambda_smooth_star, best_cv_bce, rows

# Step 6 — Define the SNB-GAM trainer

This step defines the smooth Net Benefit fine-tuning procedure for the GAM. The SNB-GAM is initialized from the fitted BCE-GAM and then optimized with Adam using an annealing schedule over the smooth Net Benefit temperature. The same GAM regularization terms selected during BCE training are retained during SNB fine-tuning. No validation set is used; early stopping is based on the hard Net Benefit evaluated on the training set.

In [12]:
from nbloss.trainer import nb_anneal_only_with_l2


def make_gam_penalty_fn(
    *,
    lambda_lin: float,
    lambda_smooth: float,
    non_spline_idx,
    spline_block_slices,
):
    """
    Creates the GAM-specific penalty function used by the generic SNB trainer.

    Important:
    - The trainer already multiplies penalty_fn(model) by l2_lambda.
    - Therefore, when calling the trainer, set l2_lambda=1.0.
    - Bias is not penalized as long as the underlying penalty functions use
      model.linear.weight rather than model.parameters().
    """

    def gam_penalty_fn(model: nn.Module) -> torch.Tensor:
        pen_lin = _linear_l2_penalty_from_idx(
            model,
            non_spline_idx,
        )

        pen_smooth = _smoothness_penalty_from_blocks(
            model,
            spline_block_slices,
        )

        return (
            float(lambda_lin) * pen_lin
            + float(lambda_smooth) * pen_smooth
        )

    return gam_penalty_fn

# Step 7 — Create raw Framingham train/test splits

This step creates the five rotating 80/20 train/test splits used for the Framingham GAM experiment. The splits are created once on the raw Framingham data and reused for both preprocessing variants. GAM preprocessing is applied later inside the runner, separately for `mild` and `moderate`, and is always fitted on the training data only.

In [20]:
# ============================================================
# Create Framingham GAM splits
# New 15-year incident ANYCHD analysis
# ============================================================

splits_mild = make_framingham_gam_splits_5x(
    model_df,
    target_col="FifteenYearCHD",
    base_seed=42,
    n_runs=5,
    fe_level="mild",
    spline_n_knots=8,
    spline_degree=3,
)

splits_moderate = make_framingham_gam_splits_5x(
    model_df,
    target_col="FifteenYearCHD",
    base_seed=42,
    n_runs=5,
    fe_level="moderate",
    spline_n_knots=8,
    spline_degree=3,
)


# Step 8 — Define local calibration helpers

This step defines helper functions for local post-hoc calibration of the BCE-GAM. As in the logistic-regression experiment, local calibration is fitted only on training observations whose original BCE-GAM predicted probabilities fall near the relevant clinical threshold. Temperature scaling and Platt scaling are then applied only to test observations whose original BCE-GAM probabilities fall in the same probability range. No calibration is applied after SNB-GAM training.

In [14]:
def sigmoid_np(logits_np: np.ndarray) -> np.ndarray:
    logits_t = torch.tensor(np.asarray(logits_np, dtype=np.float32).reshape(-1))
    return torch.sigmoid(logits_t).detach().cpu().numpy().astype(np.float64)


@torch.no_grad()
def predict_logits_torch(
    model: nn.Module,
    X_np: np.ndarray,
    *,
    device: str = DEVICE,
) -> np.ndarray:
    device_t = torch.device(device)

    model.eval()

    X_t = torch.tensor(
        X_np,
        dtype=torch.float32,
        device=device_t,
    )

    return (
        model(X_t)
        .detach()
        .cpu()
        .view(-1)
        .numpy()
        .astype(np.float64)
    )


def subset_counts(mask, y_np):
    y_sub = np.asarray(y_np).reshape(-1)[np.asarray(mask, dtype=bool)]

    n_pos = int(np.sum(y_sub == 1))
    n_neg = int(np.sum(y_sub == 0))

    return int(len(y_sub)), n_pos, n_neg


def find_local_calibration_range(
    probs_train: np.ndarray,
    y_train: np.ndarray,
    *,
    t_ref: float,
    start_half_width: float = LOCAL_START_HALF_WIDTH,
    expand_step: float = LOCAL_EXPAND_STEP,
    min_pos: int = LOCAL_MIN_POS,
    min_neg: int = LOCAL_MIN_NEG,
) -> dict:
    probs_train = np.asarray(probs_train, dtype=float).reshape(-1)
    y_train = np.asarray(y_train).reshape(-1)

    half_width = float(start_half_width)
    n_expand_steps = 0

    while True:
        low = max(0.0, float(t_ref) - half_width)
        high = min(1.0, float(t_ref) + half_width)

        mask = (probs_train >= low) & (probs_train <= high)
        n, n_pos, n_neg = subset_counts(mask, y_train)

        met_minimum = (n_pos >= int(min_pos)) and (n_neg >= int(min_neg))
        used_full_range = (low <= 0.0) and (high >= 1.0)

        if met_minimum or used_full_range:
            return {
                "low": float(low),
                "high": float(high),
                "half_width": float(half_width),
                "range_width": float(high - low),
                "n_expand_steps": int(n_expand_steps),
                "n": int(n),
                "n_pos": int(n_pos),
                "n_neg": int(n_neg),
                "met_minimum": bool(met_minimum),
                "used_full_range": bool(used_full_range),
                "mask": mask,
            }

        half_width += float(expand_step)
        n_expand_steps += 1


def fit_temperature_from_logits_np(
    logits_np: np.ndarray,
    y_np: np.ndarray,
    *,
    max_iter: int = TEMP_MAX_ITER,
    device: str = DEVICE,
) -> dict:
    device_t = torch.device(device)

    logits = torch.tensor(
        np.asarray(logits_np, dtype=np.float32).reshape(-1),
        device=device_t,
    )

    y = torch.tensor(
        np.asarray(y_np, dtype=np.float32).reshape(-1),
        device=device_t,
    )

    bce = nn.BCEWithLogitsLoss(reduction="mean")

    with torch.no_grad():
        nll_before = float(bce(logits, y).detach().cpu().item())

    log_temperature = torch.zeros(
        (),
        dtype=torch.float32,
        device=device_t,
        requires_grad=True,
    )

    optimizer = torch.optim.LBFGS(
        [log_temperature],
        lr=0.1,
        max_iter=int(max_iter),
        line_search_fn="strong_wolfe",
    )

    def closure():
        optimizer.zero_grad(set_to_none=True)

        temperature = torch.exp(log_temperature).clamp(
            min=1e-6,
            max=1e6,
        )

        loss = bce(logits / temperature, y)

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite temperature loss: {loss.detach().item()}"
            )

        loss.backward()
        return loss

    optimizer.step(closure)

    with torch.no_grad():
        temperature = torch.exp(log_temperature).clamp(
            min=1e-6,
            max=1e6,
        )

        nll_after = float(
            bce(logits / temperature, y)
            .detach()
            .cpu()
            .item()
        )

    return {
        "temperature": float(temperature.detach().cpu().item()),
        "nll_before": nll_before,
        "nll_after": nll_after,
    }


def apply_local_temperature_to_logits(
    logits_np: np.ndarray,
    probs_reference_np: np.ndarray,
    *,
    low: float,
    high: float,
    temperature: float,
) -> tuple[np.ndarray, np.ndarray]:
    logits_out = np.asarray(logits_np, dtype=np.float64).reshape(-1).copy()
    probs_reference_np = np.asarray(probs_reference_np, dtype=float).reshape(-1)

    mask = (
        (probs_reference_np >= float(low))
        & (probs_reference_np <= float(high))
    )

    logits_out[mask] = logits_out[mask] / float(temperature)

    return logits_out, mask


def fit_platt_from_logits_np(
    logits_np: np.ndarray,
    y_np: np.ndarray,
    *,
    max_iter: int = PLATT_MAX_ITER,
    device: str = DEVICE,
) -> dict:
    device_t = torch.device(device)

    logits = torch.tensor(
        np.asarray(logits_np, dtype=np.float32).reshape(-1),
        device=device_t,
    )

    y = torch.tensor(
        np.asarray(y_np, dtype=np.float32).reshape(-1),
        device=device_t,
    )

    bce = nn.BCEWithLogitsLoss(reduction="mean")

    with torch.no_grad():
        nll_before = float(bce(logits, y).detach().cpu().item())

    slope = torch.ones(
        (),
        dtype=torch.float32,
        device=device_t,
        requires_grad=True,
    )

    intercept = torch.zeros(
        (),
        dtype=torch.float32,
        device=device_t,
        requires_grad=True,
    )

    optimizer = torch.optim.LBFGS(
        [slope, intercept],
        lr=0.1,
        max_iter=int(max_iter),
        line_search_fn="strong_wolfe",
    )

    def closure():
        optimizer.zero_grad(set_to_none=True)

        calibrated_logits = slope * logits + intercept
        loss = bce(calibrated_logits, y)

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite Platt loss: {loss.detach().item()}"
            )

        loss.backward()
        return loss

    optimizer.step(closure)

    with torch.no_grad():
        calibrated_logits = slope * logits + intercept

        nll_after = float(
            bce(calibrated_logits, y)
            .detach()
            .cpu()
            .item()
        )

    return {
        "platt_slope": float(slope.detach().cpu().item()),
        "platt_intercept": float(intercept.detach().cpu().item()),
        "nll_before": nll_before,
        "nll_after": nll_after,
    }


def apply_local_platt_to_logits(
    logits_np: np.ndarray,
    probs_reference_np: np.ndarray,
    *,
    low: float,
    high: float,
    slope: float,
    intercept: float,
) -> tuple[np.ndarray, np.ndarray]:
    logits_out = np.asarray(logits_np, dtype=np.float64).reshape(-1).copy()
    probs_reference_np = np.asarray(probs_reference_np, dtype=float).reshape(-1)

    mask = (
        (probs_reference_np >= float(low))
        & (probs_reference_np <= float(high))
    )

    logits_out[mask] = (
        float(slope) * logits_out[mask]
        + float(intercept)
    )

    return logits_out, mask


def evaluate_nb_from_logits_np(
    logits_np: np.ndarray,
    y_np: np.ndarray,
    *,
    thresh_min: float,
    thresh_max: float,
    num_points: int = TEST_RANGE_POINTS,
) -> float:
    logits_t = torch.tensor(
        np.asarray(logits_np, dtype=np.float32).reshape(-1)
    )

    y_t = torch.tensor(
        np.asarray(y_np, dtype=np.float32).reshape(-1)
    )

    return float(
        average_nb_over_range(
            logits_t,
            y_t,
            thresh_min=float(thresh_min),
            thresh_max=float(thresh_max),
            num_points=int(num_points),
            input_is_logit=True,
            method="mean",
        )
    )

# Step 9 — Run the Framingham GAM experiment

This step runs the complete Framingham generalized additive model (GAM) experiment across the five repeated 80/20 train/test splits and both preprocessing variants (`mild` and `moderate`). For each split, the BCE-GAM first selects the linear and smoothness penalties by inner cross-validation on the training data only and is then refitted on the full training set. Local temperature scaling and local Platt scaling are subsequently fitted using only the subset of training observations whose predicted risks lie near the relevant clinical threshold and are applied only to test observations in the same probability range. Finally, the SNB-GAM is warm-started from the fitted BCE-GAM and fine-tuned using the smooth Net Benefit objective. The resulting fold-level performance metrics are stored in `results_df` and are used later to create summary tables and paired statistical comparisons.

In [21]:
from copy import deepcopy

from nbloss.trainer import nb_anneal_only_with_l2


LAMBDA_LIN_GRID = (0.0, 1e-4, 1e-3, 1e-2)
LAMBDA_SMOOTH_GRID = (1e-4, 1e-3, 1e-2, 1e-1)

DATASETS = {
    "mild": splits_mild,
    "moderate": splits_moderate,
}

CLINICAL_BANDS = [
    ("t_0.05", 0.05, *band_from_t_ref(0.05)),
    ("t_0.10", 0.10, *band_from_t_ref(0.10)),
    ("t_0.20", 0.20, *band_from_t_ref(0.20)),
]

results = []


def add_result_row_gam(
    *,
    dataset_name: str,
    run_id: int,
    band_name: str,
    model_name: str,
    split_pack: dict,
    t_ref: float,
    t_min: float,
    t_max: float,
    lambda_lin: float,
    lambda_smooth: float,
    cv_bce: float,
    train_bce: float,
    test_nb: float,
    delta_vs_bce: float,
    local_info: dict | None = None,
    calibration_info: dict | None = None,
):
    local_info = local_info or {}
    calibration_info = calibration_info or {}

    results.append(
        {
            "dataset": dataset_name,
            "run_id": int(run_id),
            "band_name": band_name,
            "model": model_name,
            "prev_train": float(split_pack["prev_train"]),
            "prev_test": float(split_pack["prev_test"]),
            "t_ref": float(t_ref),
            "t_min": float(t_min),
            "t_max": float(t_max),
            "lambda_lin": float(lambda_lin),
            "lambda_smooth": float(lambda_smooth),
            "cv_bce": float(cv_bce),
            "train_bce": float(train_bce),
            "test_nb": float(test_nb),
            "delta_vs_bce": float(delta_vs_bce),
            "local_low": local_info.get("low", np.nan),
            "local_high": local_info.get("high", np.nan),
            "local_half_width": local_info.get("half_width", np.nan),
            "local_range_width": local_info.get("range_width", np.nan),
            "local_expand_steps": local_info.get("n_expand_steps", np.nan),
            "local_train_n": local_info.get("n", np.nan),
            "local_train_pos": local_info.get("n_pos", np.nan),
            "local_train_neg": local_info.get("n_neg", np.nan),
            "local_met_minimum": local_info.get("met_minimum", np.nan),
            "local_used_full_range": local_info.get("used_full_range", np.nan),
            "temperature": calibration_info.get("temperature", np.nan),
            "platt_slope": calibration_info.get("platt_slope", np.nan),
            "platt_intercept": calibration_info.get("platt_intercept", np.nan),
            "train_local_nll_before": calibration_info.get("nll_before", np.nan),
            "train_local_nll_after": calibration_info.get("nll_after", np.nan),
            "phi_dim": int(split_pack["Phi_tr"].shape[1]),
        }
    )


def make_gam_penalty_fn(
    *,
    lambda_lin: float,
    lambda_smooth: float,
    non_spline_idx,
    spline_block_slices,
):
    def gam_penalty_fn(model: nn.Module) -> torch.Tensor:
        pen_lin = _linear_l2_penalty_from_idx(
            model,
            non_spline_idx,
        )

        pen_smooth = _smoothness_penalty_from_blocks(
            model,
            spline_block_slices,
        )

        return (
            float(lambda_lin) * pen_lin
            + float(lambda_smooth) * pen_smooth
        )

    return gam_penalty_fn


for dataset_name, splits_list in DATASETS.items():
    print(f"\n==================== DATASET: {dataset_name} ====================")

    for sp in splits_list:
        run_id = int(sp["run_id"])
        split_seed = int(sp["seed"])

        Phi_tr = np.asarray(sp["Phi_tr"], dtype=np.float32)
        Phi_te = np.asarray(sp["Phi_te"], dtype=np.float32)

        y_tr = np.asarray(sp["y_train"], dtype=np.float32).reshape(-1)
        y_te = np.asarray(sp["y_test"], dtype=np.float32).reshape(-1)

        non_spline_idx = sp["non_spline_idx"]
        spline_block_slices = sp["spline_block_slices"]

        d_in_phi = int(Phi_tr.shape[1])

        def make_gam_model():
            return TorchGAM(d_in_phi)

        lambda_lin_star, lambda_smooth_star, best_cv_bce, cv_rows = (
            select_gam_penalties_by_cv_bce(
                LAMBDA_LIN_GRID,
                LAMBDA_SMOOTH_GRID,
                make_model_fn=make_gam_model,
                X_tr=Phi_tr,
                y_tr=y_tr,
                non_spline_idx=non_spline_idx,
                spline_block_slices=spline_block_slices,
                device=DEVICE,
                n_splits=5,
                lbfgs_max_iter=500,
                seed=split_seed,
            )
        )

        print(
            f"\n[{dataset_name} | run {run_id}] "
            f"prev_train={float(y_tr.mean()):.4f} | "
            f"lambda_lin={lambda_lin_star:g} | "
            f"lambda_smooth={lambda_smooth_star:g} | "
            f"best inner-CV BCE={best_cv_bce:.6f}"
        )

        train_dl = make_loader(
            Phi_tr,
            y_tr,
            batch=BATCH,
            shuffle=True,
            seed=split_seed,
        )

        bce_model, train_bce = fit_bce_gam_lbfgs(
            make_gam_model,
            Phi_tr,
            y_tr,
            lambda_lin=float(lambda_lin_star),
            lambda_smooth=float(lambda_smooth_star),
            non_spline_idx=non_spline_idx,
            spline_block_slices=spline_block_slices,
            max_iter=500,
            device=DEVICE,
        )

        logits_bce_train = predict_logits_torch(
            bce_model,
            Phi_tr,
            device=DEVICE,
        )

        logits_bce_test = predict_logits_torch(
            bce_model,
            Phi_te,
            device=DEVICE,
        )

        probs_bce_train = sigmoid_np(logits_bce_train)
        probs_bce_test = sigmoid_np(logits_bce_test)

        for band_name, t_ref, t_min, t_max in CLINICAL_BANDS:
            print(
                f"\n[{dataset_name} | run {run_id}] "
                f"=== BAND: {band_name} "
                f"(t_ref={t_ref:.4f}, [{t_min:.4f}, {t_max:.4f}]) ==="
            )

            nb_bce = evaluate_nb_from_logits_np(
                logits_bce_test,
                y_te,
                thresh_min=t_min,
                thresh_max=t_max,
                num_points=TEST_RANGE_POINTS,
            )

            print(f"[TEST][{band_name}] BCE-GAM NB={nb_bce:.6f}")

            local_bce = find_local_calibration_range(
                probs_bce_train,
                y_tr,
                t_ref=float(t_ref),
                start_half_width=LOCAL_START_HALF_WIDTH,
                expand_step=LOCAL_EXPAND_STEP,
                min_pos=LOCAL_MIN_POS,
                min_neg=LOCAL_MIN_NEG,
            )

            logits_bce_train_local = logits_bce_train[local_bce["mask"]]
            ytr_bce_local = y_tr[local_bce["mask"]]

            temp_bce_fit = fit_temperature_from_logits_np(
                logits_bce_train_local,
                ytr_bce_local,
                max_iter=TEMP_MAX_ITER,
                device=DEVICE,
            )

            platt_bce_fit = fit_platt_from_logits_np(
                logits_bce_train_local,
                ytr_bce_local,
                max_iter=PLATT_MAX_ITER,
                device=DEVICE,
            )

            logits_bce_temp_test, _ = apply_local_temperature_to_logits(
                logits_bce_test,
                probs_bce_test,
                low=local_bce["low"],
                high=local_bce["high"],
                temperature=temp_bce_fit["temperature"],
            )

            logits_bce_platt_test, _ = apply_local_platt_to_logits(
                logits_bce_test,
                probs_bce_test,
                low=local_bce["low"],
                high=local_bce["high"],
                slope=platt_bce_fit["platt_slope"],
                intercept=platt_bce_fit["platt_intercept"],
            )

            nb_bce_temp = evaluate_nb_from_logits_np(
                logits_bce_temp_test,
                y_te,
                thresh_min=t_min,
                thresh_max=t_max,
                num_points=TEST_RANGE_POINTS,
            )

            nb_bce_platt = evaluate_nb_from_logits_np(
                logits_bce_platt_test,
                y_te,
                thresh_min=t_min,
                thresh_max=t_max,
                num_points=TEST_RANGE_POINTS,
            )

            snb_start = TorchGAM(d_in_phi).to(DEVICE)
            snb_start.load_state_dict(
                {
                    k: v.detach().cpu().clone()
                    for k, v in bce_model.state_dict().items()
                }
            )

            gam_penalty_fn = make_gam_penalty_fn(
                lambda_lin=float(lambda_lin_star),
                lambda_smooth=float(lambda_smooth_star),
                non_spline_idx=non_spline_idx,
                spline_block_slices=spline_block_slices,
            )

            snb_model = nb_anneal_only_with_l2(
                model=snb_start,
                train_dl=train_dl,
                thresh_min=float(t_min),
                thresh_max=float(t_max),
                num_points_train=int(TRAIN_RANGE_POINTS),
                inverse_temps=tuple(INVERSE_TEMPS),
                epochs_per_temp=int(EPOCHS_PER_TEMP),
                patience_hard=int(PATIENCE_HARD),
                hard_range_num_points=int(TEST_RANGE_POINTS),
                lr_adam=float(LR_ADAMW),
                l2_lambda=1.0,
                penalty_fn=gam_penalty_fn,
                device=DEVICE,
                seed=int(MODEL_SEED) + 1000 * run_id + 17,
                log_every=20,
            )

            logits_snb_test = predict_logits_torch(
                snb_model,
                Phi_te,
                device=DEVICE,
            )

            nb_snb = evaluate_nb_from_logits_np(
                logits_snb_test,
                y_te,
                thresh_min=t_min,
                thresh_max=t_max,
                num_points=TEST_RANGE_POINTS,
            )

            add_result_row_gam(
                dataset_name=dataset_name,
                run_id=run_id,
                band_name=band_name,
                model_name="bce_gam",
                split_pack=sp,
                t_ref=t_ref,
                t_min=t_min,
                t_max=t_max,
                lambda_lin=lambda_lin_star,
                lambda_smooth=lambda_smooth_star,
                cv_bce=best_cv_bce,
                train_bce=train_bce,
                test_nb=nb_bce,
                delta_vs_bce=0.0,
            )

            add_result_row_gam(
                dataset_name=dataset_name,
                run_id=run_id,
                band_name=band_name,
                model_name="bce_gam_local_temperature",
                split_pack=sp,
                t_ref=t_ref,
                t_min=t_min,
                t_max=t_max,
                lambda_lin=lambda_lin_star,
                lambda_smooth=lambda_smooth_star,
                cv_bce=best_cv_bce,
                train_bce=train_bce,
                test_nb=nb_bce_temp,
                delta_vs_bce=nb_bce_temp - nb_bce,
                local_info=local_bce,
                calibration_info=temp_bce_fit,
            )

            add_result_row_gam(
                dataset_name=dataset_name,
                run_id=run_id,
                band_name=band_name,
                model_name="bce_gam_local_platt",
                split_pack=sp,
                t_ref=t_ref,
                t_min=t_min,
                t_max=t_max,
                lambda_lin=lambda_lin_star,
                lambda_smooth=lambda_smooth_star,
                cv_bce=best_cv_bce,
                train_bce=train_bce,
                test_nb=nb_bce_platt,
                delta_vs_bce=nb_bce_platt - nb_bce,
                local_info=local_bce,
                calibration_info=platt_bce_fit,
            )

            add_result_row_gam(
                dataset_name=dataset_name,
                run_id=run_id,
                band_name=band_name,
                model_name="snb_gam",
                split_pack=sp,
                t_ref=t_ref,
                t_min=t_min,
                t_max=t_max,
                lambda_lin=lambda_lin_star,
                lambda_smooth=lambda_smooth_star,
                cv_bce=best_cv_bce,
                train_bce=train_bce,
                test_nb=nb_snb,
                delta_vs_bce=nb_snb - nb_bce,
            )

            print(
                f"[CAL][{band_name}] "
                f"BCE-temp Δ={nb_bce_temp - nb_bce:+.6f} | "
                f"BCE-Platt Δ={nb_bce_platt - nb_bce:+.6f} | "
                f"SNB Δ={nb_snb - nb_bce:+.6f}"
            )

        if DEVICE == "cuda":
            torch.cuda.empty_cache()

results_df = pd.DataFrame(results)

print("\nFinished.")
print("results_df shape:", results_df.shape)
display(results_df.head())


==================== DATASET: mild ====================

[mild | run 0] prev_train=0.1667 | lambda_lin=0.0001 | lambda_smooth=0.1 | best inner-CV BCE=0.388251

[mild | run 0] === BAND: t_0.05 (t_ref=0.0500, [0.0250, 0.0750]) ===
[TEST][t_0.05] BCE-GAM NB=0.127166
[SNB start] initial train hard NB range = 0.127774
[SNB inverse_temp=1] epoch 020 | train_loss=-0.122096 | train_hard_nb=0.122602
>>> Early stop inverse-temperature phase.
[commit] inverse_temp=1 no improvement greater than epsilon_nb=1e-12; keep global train hard NB: 0.127774
[SNB inverse_temp=4] epoch 020 | train_loss=-0.126691 | train_hard_nb=0.128162
[SNB inverse_temp=4] epoch 040 | train_loss=-0.126899 | train_hard_nb=0.128402
[SNB inverse_temp=4] epoch 060 | train_loss=-0.127015 | train_hard_nb=0.128518
>>> Early stop inverse-temperature phase.
[commit] inverse_temp=4 improved global train hard NB: 0.127774 → 0.128529
[SNB inverse_temp=10] epoch 020 | train_loss=-0.128303 | train_hard_nb=0.128971
[SNB inverse_temp=10] e

,dataset,run_id,band_name,model,prev_train,prev_test,t_ref,t_min,t_max,lambda_lin,...,local_train_pos,local_train_neg,local_met_minimum,local_used_full_range,temperature,platt_slope,platt_intercept,train_local_nll_before,train_local_nll_after,phi_dim
0,mild,0,t_0.05,bce_gam,0.166667,0.166446,0.05,0.025,0.075,0.0001,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,79
1,mild,0,t_0.05,bce_gam_local_temperature,0.166667,0.166446,0.05,0.025,0.075,0.0001,...,120.0,1625.0,True,False,0.973172,NaN,NaN,0.238955,0.238806,79
2,mild,0,t_0.05,bce_gam_local_platt,0.166667,0.166446,0.05,0.025,0.075,0.0001,...,120.0,1625.0,True,False,NaN,1.02401,-0.008966,0.238955,0.238806,79
3,mild,0,t_0.05,snb_gam,0.166667,0.166446,0.05,0.025,0.075,0.0001,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,79
4,mild,0,t_0.10,bce_gam,0.166667,0.166446,0.10,0.075,0.125,0.0001,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,79


# Step 10 — Summarize the Framingham GAM results

This step summarizes the results of the Framingham GAM experiment. First, the fold-level results stored in `results_df` are reshaped to create paired comparisons between each model and the baseline BCE-GAM within the same preprocessing variant, train/test split, and clinical threshold band. Paired t-tests are then performed to evaluate whether local temperature scaling, local Platt scaling, or SNB fine-tuning lead to systematic improvements in Net Benefit relative to the BCE-GAM.

Two summary tables are created:

- `summary_tests_df` contains the paired statistical comparisons, including the mean and standard deviation of the change in Net Benefit, the number of wins, losses, and ties relative to the BCE-GAM, and the paired t-test results.

- `summary_df` contains aggregated performance summaries across the five runs, including the mean and standard deviation of Net Benefit, local calibration statistics, the selected GAM penalty parameters, and the average cross-validated BCE value.

These summaries are subsequently used to create the final tables and figures for the Framingham GAM experiments.

In [22]:
from scipy.stats import ttest_rel


print("\n======================================")
print("Finished. results_df shape:")
print(results_df.shape)
print("======================================")
display(results_df.head())


paired_df = (
    results_df
    .pivot_table(
        index=["dataset", "run_id", "band_name"],
        columns="model",
        values="test_nb",
        aggfunc="first",
    )
    .reset_index()
)

baseline_model = "bce_gam"

summary_rows = []

for (dataset_name, band_name), sub in paired_df.groupby(
    ["dataset", "band_name"]
):
    for model_name in [
        "bce_gam_local_temperature",
        "bce_gam_local_platt",
        "snb_gam",
    ]:
        if model_name not in sub.columns:
            continue

        tmp = sub.dropna(
            subset=[baseline_model, model_name]
        ).copy()

        n_runs = len(tmp)

        if n_runs == 0:
            continue

        delta = tmp[model_name] - tmp[baseline_model]

        if n_runs >= 2:
            t_stat, p_value = ttest_rel(
                tmp[model_name],
                tmp[baseline_model],
            )

            t_stat = float(t_stat)
            p_value = float(p_value)
        else:
            t_stat = np.nan
            p_value = np.nan

        summary_rows.append(
            {
                "dataset": dataset_name,
                "band_name": band_name,
                "model": model_name,
                "n_runs": int(n_runs),
                "mean_bce_gam": float(
                    tmp[baseline_model].mean()
                ),
                "mean_model_nb": float(
                    tmp[model_name].mean()
                ),
                "mean_delta_vs_bce": float(
                    delta.mean()
                ),
                "sd_delta_vs_bce": (
                    float(delta.std(ddof=1))
                    if n_runs > 1
                    else np.nan
                ),
                "wins_vs_bce": int((delta > 0).sum()),
                "losses_vs_bce": int((delta < 0).sum()),
                "ties_vs_bce": int((delta == 0).sum()),
                "paired_t_vs_bce": t_stat,
                "paired_p_vs_bce": p_value,
                "significant_0.05": (
                    bool(p_value < 0.05)
                    if np.isfinite(p_value)
                    else False
                ),
            }
        )

summary_tests_df = pd.DataFrame(summary_rows)

print("\n======================================")
print("Paired summary versus BCE-GAM across 5 runs")
print("======================================")
display(summary_tests_df)


summary_df = (
    results_df
    .groupby(
        ["dataset", "band_name", "model"],
        as_index=False,
    )
    .agg(
        mean_test_nb=("test_nb", "mean"),
        sd_test_nb=("test_nb", "std"),
        mean_delta_vs_bce=("delta_vs_bce", "mean"),
        sd_delta_vs_bce=("delta_vs_bce", "std"),
        mean_local_low=("local_low", "mean"),
        mean_local_high=("local_high", "mean"),
        mean_local_range_width=("local_range_width", "mean"),
        mean_local_train_n=("local_train_n", "mean"),
        mean_local_train_pos=("local_train_pos", "mean"),
        mean_local_train_neg=("local_train_neg", "mean"),
        mean_temperature=("temperature", "mean"),
        mean_platt_slope=("platt_slope", "mean"),
        mean_platt_intercept=("platt_intercept", "mean"),
        mean_train_local_nll_before=(
            "train_local_nll_before",
            "mean",
        ),
        mean_train_local_nll_after=(
            "train_local_nll_after",
            "mean",
        ),
        mean_lambda_lin=("lambda_lin", "mean"),
        mean_lambda_smooth=("lambda_smooth", "mean"),
        mean_cv_bce=("cv_bce", "mean"),
        n_runs=("run_id", "nunique"),
    )
)

print("\n======================================")
print("Mean results across 5 runs")
print("======================================")
display(summary_df)


Finished. results_df shape:
(120, 31)


,dataset,run_id,band_name,model,prev_train,prev_test,t_ref,t_min,t_max,lambda_lin,...,local_train_pos,local_train_neg,local_met_minimum,local_used_full_range,temperature,platt_slope,platt_intercept,train_local_nll_before,train_local_nll_after,phi_dim
0,mild,0,t_0.05,bce_gam,0.166667,0.166446,0.05,0.025,0.075,0.0001,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,79
1,mild,0,t_0.05,bce_gam_local_temperature,0.166667,0.166446,0.05,0.025,0.075,0.0001,...,120.0,1625.0,True,False,0.973172,NaN,NaN,0.238955,0.238806,79
2,mild,0,t_0.05,bce_gam_local_platt,0.166667,0.166446,0.05,0.025,0.075,0.0001,...,120.0,1625.0,True,False,NaN,1.02401,-0.008966,0.238955,0.238806,79
3,mild,0,t_0.05,snb_gam,0.166667,0.166446,0.05,0.025,0.075,0.0001,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,79
4,mild,0,t_0.10,bce_gam,0.166667,0.166446,0.10,0.075,0.125,0.0001,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,79



Paired summary versus BCE-GAM across 5 runs


,dataset,band_name,model,n_runs,mean_bce_gam,mean_model_nb,mean_delta_vs_bce,sd_delta_vs_bce,wins_vs_bce,losses_vs_bce,ties_vs_bce,paired_t_vs_bce,paired_p_vs_bce,significant_0.05
0,mild,t_0.05,bce_gam_local_temperature,5,0.127292,0.127478,0.000186,0.000153,5,0,0,2.704273,0.053856,False
1,mild,t_0.05,bce_gam_local_platt,5,0.127292,0.127539,0.000247,0.000208,5,0,0,2.654563,0.056711,False
2,mild,t_0.05,snb_gam,5,0.127292,0.126907,-0.000385,0.000846,1,4,0,-1.017252,0.366558,False
3,mild,t_0.10,bce_gam_local_temperature,5,0.096803,0.096783,-0.000020,0.000432,2,3,0,-0.101689,0.923897,False
4,mild,t_0.10,bce_gam_local_platt,5,0.096803,0.096860,0.000057,0.000397,3,2,0,0.320473,0.764653,False
5,mild,t_0.10,snb_gam,5,0.096803,0.095242,-0.001561,0.002049,1,4,0,-1.703272,0.163726,False
6,mild,t_0.20,bce_gam_local_temperature,5,0.051729,0.051549,-0.000180,0.000552,2,3,0,-0.729524,0.506104,False
7,mild,t_0.20,bce_gam_local_platt,5,0.051729,0.051455,-0.000274,0.000852,2,3,0,-0.719759,0.511485,False
8,mild,t_0.20,snb_gam,5,0.051729,0.047821,-0.003908,0.004355,1,4,0,-2.006549,0.115252,False
9,moderate,t_0.05,bce_gam_local_temperature,5,0.127101,0.127343,0.000242,0.000178,5,0,0,3.046111,0.038173,True



Mean results across 5 runs


,dataset,band_name,model,mean_test_nb,sd_test_nb,mean_delta_vs_bce,sd_delta_vs_bce,mean_local_low,mean_local_high,mean_local_range_width,...,mean_local_train_neg,mean_temperature,mean_platt_slope,mean_platt_intercept,mean_train_local_nll_before,mean_train_local_nll_after,mean_lambda_lin,mean_lambda_smooth,mean_cv_bce,n_runs
0,mild,t_0.05,bce_gam,0.127292,0.001266,0.000000,0.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.00082,0.1,0.390359,5
1,mild,t_0.05,bce_gam_local_platt,0.127539,0.001447,0.000247,0.000208,0.0,0.15,0.15,...,1601.0,NaN,1.105898,0.195622,0.244329,0.244068,0.00082,0.1,0.390359,5
2,mild,t_0.05,bce_gam_local_temperature,0.127478,0.001374,0.000186,0.000153,0.0,0.15,0.15,...,1601.0,0.975097,NaN,NaN,0.244329,0.244175,0.00082,0.1,0.390359,5
3,mild,t_0.05,snb_gam,0.126907,0.002076,-0.000385,0.000846,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.00082,0.1,0.390359,5
4,mild,t_0.10,bce_gam,0.096803,0.004191,0.000000,0.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.00082,0.1,0.390359,5
5,mild,t_0.10,bce_gam_local_platt,0.096860,0.004381,0.000057,0.000397,0.0,0.20,0.20,...,1911.0,NaN,1.105550,0.193121,0.283944,0.283684,0.00082,0.1,0.390359,5
6,mild,t_0.10,bce_gam_local_temperature,0.096783,0.004255,-0.000020,0.000432,0.0,0.20,0.20,...,1911.0,0.982039,NaN,NaN,0.283944,0.283860,0.00082,0.1,0.390359,5
7,mild,t_0.10,snb_gam,0.095242,0.004656,-0.001561,0.002049,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.00082,0.1,0.390359,5
8,mild,t_0.20,bce_gam,0.051729,0.005545,0.000000,0.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.00082,0.1,0.390359,5
9,mild,t_0.20,bce_gam_local_platt,0.051455,0.004916,-0.000274,0.000852,0.1,0.30,0.20,...,1107.4,NaN,0.973441,-0.021765,0.465051,0.464883,0.00082,0.1,0.390359,5
